# 🧩 Deep Learning & Exact Sudoku AI Solver

This interactive notebook demonstrates how to build and train a **Deep Learning Convolutional Neural Network (CNN)** in PyTorch to solve Sudoku puzzles, alongside an **Exact Backtracking Solver** benchmark.

### 📊 Verified Online Datasets
- [**Kaggle 1 Million Sudoku Games (Bryan Park)**](https://www.kaggle.com/datasets/bryanpark/sudoku): 1,000,000 puzzles and solutions.
- [**Kaggle 3 Million Puzzles with Ratings (Radcliffe)**](https://www.kaggle.com/datasets/radcliffe/3-million-sudoku-puzzles-with-ratings): 3,000,000 puzzles with difficulty ratings.

---

## 1. Imports & Device Setup

In [ ]:
import os
import csv
import time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using compute device: {device}")

## 2. Board Representation & One-Hot Encoding

Numbers in Sudoku are **categorical symbols**, not arithmetic quantities. 
We convert each $9 \times 9$ board into a $(9 \text{ channels}, 9 \text{ rows}, 9 \text{ cols})$ binary one-hot tensor.

In [ ]:
def print_board(board: np.ndarray) -> None:
    """Prints a 9x9 board with 3x3 block dividers."""
    for i in range(9):
        if i % 3 == 0 and i != 0:
            print("- - - - - - - - - - - - -")
        row_str = []
        for j in range(9):
            if j % 3 == 0 and j != 0:
                row_str.append("|")
            val = board[i, j]
            row_str.append(str(val) if val != 0 else ".")
        print(" ".join(row_str))
    print()

def to_one_hot(board: np.ndarray) -> np.ndarray:
    """Converts 9x9 board (0..9) into (9, 9, 9) binary float32 tensor."""
    tensor = np.zeros((9, 9, 9), dtype=np.float32)
    for num in range(1, 10):
        tensor[num - 1] = (board == num).astype(np.float32)
    return tensor

def from_one_hot(tensor: np.ndarray) -> np.ndarray:
    """Converts (9, 9, 9) tensor back into 9x9 board."""
    has_digit = np.any(tensor > 0.5, axis=0)
    digits = np.argmax(tensor, axis=0) + 1
    return np.where(has_digit, digits, 0).astype(int)

def is_valid_move(board: np.ndarray, row: int, col: int, num: int) -> bool:
    """Checks Row, Column, and 3x3 Box constraints."""
    if num in board[row, :]: return False
    if num in board[:, col]: return False
    start_r, start_c = (row // 3) * 3, (col // 3) * 3
    if num in board[start_r:start_r+3, start_c:start_c+3]: return False
    return True

## 3. Exact Recursive Backtracking Solver (100% Benchmark)

In [ ]:
def find_empty_cell(board: np.ndarray):
    for r in range(9):
        for c in range(9):
            if board[r, c] == 0:
                return (r, c)
    return None

def solve_sudoku(board: np.ndarray) -> bool:
    """Solves Sudoku in-place using recursive backtracking."""
    empty = find_empty_cell(board)
    if empty is None:
        return True
    row, col = empty
    for num in range(1, 10):
        if is_valid_move(board, row, col, num):
            board[row, col] = num
            if solve_sudoku(board):
                return True
            board[row, col] = 0
    return False

## 4. The Deep Learning Model: SudokuCNN (9 Layers)

- Uses $3 \times 3$ Conv2D filters to mirror the $3 \times 3$ subgrid boxes.
- 9 stacked layers expand the receptive field so corner cells reach the opposite corners.

In [ ]:
class SudokuCNN(nn.Module):
    def __init__(self, num_layers: int = 9, hidden_dim: int = 64):
        super().__init__()
        layers = []
        layers.append(nn.Conv2d(9, hidden_dim, kernel_size=3, padding=1))
        layers.append(nn.BatchNorm2d(hidden_dim))
        layers.append(nn.ReLU())
        
        for _ in range(num_layers - 2):
            layers.append(nn.Conv2d(hidden_dim, hidden_dim, kernel_size=3, padding=1))
            layers.append(nn.BatchNorm2d(hidden_dim))
            layers.append(nn.ReLU())
            
        layers.append(nn.Conv2d(hidden_dim, 9, kernel_size=1))
        self.network = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.network(x)

model = SudokuCNN().to(device)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"SudokuCNN initialized with {total_params:,} parameters.")

## 5. Dataset Loader

In [ ]:
class SudokuDataset(Dataset):
    def __init__(self, csv_path: str, max_samples: int = 1000):
        self.quizzes = []
        self.solutions = []
        with open(csv_path, mode="r", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            quiz_col = next((c for c in reader.fieldnames if c.lower().strip() in ['quizzes', 'puzzle', 'quiz']), reader.fieldnames[0])
            sol_col = next((c for c in reader.fieldnames if c.lower().strip() in ['solutions', 'solution', 'sol']), reader.fieldnames[1] if len(reader.fieldnames) > 1 else reader.fieldnames[0])
            for idx, row in enumerate(reader):
                if max_samples and idx >= max_samples:
                    break
                q, s = row[quiz_col].strip(), row[sol_col].strip()
                if len(q) == 81 and len(s) == 81:
                    self.quizzes.append(q)
                    self.solutions.append(s)
        print(f"Loaded {len(self.quizzes):,} puzzle pairs from {os.path.basename(csv_path)}")

    def __len__(self):
        return len(self.quizzes)

    def __getitem__(self, idx):
        quiz_board = np.array([int(c) for c in self.quizzes[idx]], dtype=int).reshape(9, 9)
        sol_board = np.array([int(c) for c in self.solutions[idx]], dtype=int).reshape(9, 9)
        return torch.from_numpy(to_one_hot(quiz_board)), torch.from_numpy(sol_board - 1).long()

## 6. Training & Evaluation

In [ ]:
# Load sample dataset
sample_csv = "data/sample_sudoku.csv"
train_dataset = SudokuDataset(sample_csv)
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

print("Starting training for 5 epochs...")
for epoch in range(1, 6):
    model.train()
    running_loss = 0.0
    for inputs, targets in train_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    print(f"Epoch [{epoch}/5] - Average Loss: {running_loss/len(train_loader):.4f}")

## 7. Interactive Inference (DL Solver vs. Exact Solver)

In [ ]:
test_puzzle = "530070000600195000098000060800060003400803001700020006060000280000419005000080079"
board = np.array([int(c) for c in test_puzzle], dtype=int).reshape(9, 9)

print("=== ORIGINAL PUZZLE ===")
print_board(board)

# Exact Backtracking
bt_board = board.copy()
t0 = time.perf_counter()
solve_sudoku(bt_board)
bt_ms = (time.perf_counter() - t0) * 1000
print(f"=== BACKTRACKING SOLUTION ({bt_ms:.2f} ms) ===")
print_board(bt_board)

# Neural Network Forward Pass
model.eval()
tensor = torch.from_numpy(to_one_hot(board)).unsqueeze(0).to(device)
t0 = time.perf_counter()
with torch.no_grad():
    out = model(tensor)
    preds = torch.argmax(out, dim=1).squeeze(0).cpu().numpy() + 1
dl_ms = (time.perf_counter() - t0) * 1000

dl_board = np.where(board != 0, board, preds)
print(f"=== NEURAL NETWORK PREDICTION ({dl_ms:.2f} ms) ===")
print_board(dl_board)